# 9.6 Random Forest Deep Dive

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 9: Ensemble Learning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-09-ensemble-learning/09.06_random-forest-deep-dive.ipynb)

Companion code for Section 9.6 of the book. Run the cells in order from top to bottom.

**Example 9.6.1**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

data = load_breast_cancer()
X, y = data.data, data.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# ── Fit Random Forest ─────────────────────────────────────────────
rf = RandomForestClassifier(
    n_estimators=500,
    max_features='sqrt',         # √p features per split
    max_depth=None,              # Full trees
    min_samples_split=2,
    min_samples_leaf=1,
    oob_score=True,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_tr, y_tr)

print(f'RF OOB Score:          {rf.oob_score_:.4f}')
print(f'RF Test Accuracy:      {rf.score(X_te, y_te):.4f}')
print(f'RF CV Accuracy (5):    {cross_val_score(rf, X, y, cv=5).mean():.4f}')

# ── Feature Importance Types ─────────────────────────────────────
# 1. Gini importance (mean decrease in impurity)
gini_imp = rf.feature_importances_
gini_std = np.std([t.feature_importances_ for t in rf.estimators_], axis=0)

# 2. Permutation importance (more reliable, corrects for high-cardinality bias)
perm_result = permutation_importance(rf, X_te, y_te, n_repeats=20, random_state=42, n_jobs=-1)
perm_imp = perm_result.importances_mean
perm_std = perm_result.importances_std

# Compare rankings
imp_df = pd.DataFrame({
    'feature':   data.feature_names,
    'gini_imp':  gini_imp,
    'perm_imp':  perm_imp,
    'gini_std':  gini_std,
    'perm_std':  perm_std
}).sort_values('perm_imp', ascending=False)

print('\nFeature Importance (Top 10): Gini vs Permutation')
print(f'{'Feature':35s} {'Gini Imp':>10} {'Perm Imp':>10}')
print('='*60)
for _, row in imp_df.head(10).iterrows():
    print(f'{row.feature:35s} {row.gini_imp:>10.4f} {row.perm_imp:>10.4f}')

# ── Hyperparameter effects ────────────────────────────────────────
print('\nRandom Forest Hyperparameter Analysis:')
print(f'{'Parameter':25s} {'Value':>10} {'OOB':>8} {'Test':>8}')
print('='*55)

# n_estimators
for n in [10, 50, 100, 200, 500]:
    rf_n = RandomForestClassifier(n_estimators=n, oob_score=True, random_state=42, n_jobs=-1)
    rf_n.fit(X_tr, y_tr)
    print(f'{f"n_estimators={n}":25s} {n:>10} {rf_n.oob_score_:>8.4f} {rf_n.score(X_te,y_te):>8.4f}')

# max_features
print()
for mf in ['sqrt', 'log2', None, 0.3, 0.5, 0.8]:
    rf_mf = RandomForestClassifier(n_estimators=200, max_features=mf, oob_score=True,
                                    random_state=42, n_jobs=-1)
    rf_mf.fit(X_tr, y_tr)
    mf_str = str(mf)
    print(f'{f"max_features={mf_str}":25s} {str(mf):>10} {rf_mf.oob_score_:>8.4f} {rf_mf.score(X_te,y_te):>8.4f}')

# ── Comprehensive visualisation ───────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
fig.suptitle('Random Forest Deep Dive - Breast Cancer', fontsize=14, fontweight='bold')

# Feature importance: Gini
ax = axes[0,0]
top_idx = np.argsort(gini_imp)[::-1][:15]
ax.barh(range(15), gini_imp[top_idx][::-1], xerr=gini_std[top_idx][::-1],
       color='#1976D2', alpha=0.8, edgecolor='white', capsize=3)
ax.set_yticks(range(15))
ax.set_yticklabels([data.feature_names[i] for i in top_idx[::-1]], fontsize=8)
ax.set_xlabel('Mean Decrease in Impurity')
ax.set_title('Gini Feature Importance ± Std')
ax.grid(alpha=0.3, axis='x')

# Feature importance: Permutation
ax = axes[0,1]
top_perm = np.argsort(perm_imp)[::-1][:15]
ax.barh(range(15), perm_imp[top_perm][::-1], xerr=perm_std[top_perm][::-1],
       color='#43A047', alpha=0.8, edgecolor='white', capsize=3)
ax.set_yticks(range(15))
ax.set_yticklabels([data.feature_names[i] for i in top_perm[::-1]], fontsize=8)
ax.set_xlabel('Mean Decrease in Accuracy')
ax.set_title('Permutation Feature Importance ± Std')
ax.grid(alpha=0.3, axis='x')

# OOB error vs n_estimators
ax = axes[1,0]
oob_errors = []
n_range    = range(1, 201)
rf_growing = RandomForestClassifier(n_estimators=200, oob_score=True,
                                     warm_start=True, random_state=42, n_jobs=-1)
for n in n_range:
    rf_growing.n_estimators = n
    rf_growing.fit(X_tr, y_tr)
    oob_errors.append(1 - rf_growing.oob_score_)

ax.plot(n_range, oob_errors, 'b-', lw=2, alpha=0.8)
ax.set_xlabel('Number of Trees'); ax.set_ylabel('OOB Error')
ax.set_title('OOB Error vs Number of Trees\n(converges - more trees never hurt)')
ax.grid(alpha=0.3)

# max_features vs accuracy
ax = axes[1,1]
mf_values  = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0, 'sqrt', 'log2']
mf_scores  = []
for mf in mf_values:
    rf_mf = RandomForestClassifier(n_estimators=200, max_features=mf, oob_score=True,
                                    random_state=42, n_jobs=-1)
    rf_mf.fit(X_tr, y_tr)
    mf_scores.append(rf_mf.oob_score_)

mf_labels = [str(m) for m in mf_values]
ax.bar(mf_labels, mf_scores, color='#E53935', alpha=0.8, edgecolor='white')
for i, sc in enumerate(mf_scores):
    ax.text(i, sc+0.001, f'{sc:.3f}', ha='center', fontsize=9, fontweight='bold')
ax.set_xlabel('max_features'); ax.set_ylabel('OOB Accuracy')
ax.set_title('max_features Hyperparameter Effect')
ax.set_ylim(0.92, 0.975); ax.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('random_forest_deep_dive.png', dpi=150, bbox_inches='tight')
plt.show()